# Retail Rocket - Data Wrangling

A ideia desse notebook é verificar e limpar dados que possam trazer problemas para pesquisas futuras.

Será utilizada a Medallion Architecture para estruturar o pipeline de dados.
- Bronze: Dados brutos (raw ingestion)
- Silver: Dados limpos e tratados
- Gold: Métricas e análises de negócio (KPIs)


Fonte: https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset/data

In [0]:
USE retail_rocket.db;

In [0]:
CREATE OR REPLACE TABLE events_bronze
USING DELTA
AS
SELECT * 
FROM retail_rocket.db.events;

In [0]:
CREATE OR REPLACE TABLE events_silver AS
SELECT 
  visitorid,
  event,
  itemid,
  transactionid,
  timestamp,

  FROM_UNIXTIME(timestamp / 1000) AS event_time,
  YEAR(FROM_UNIXTIME(timestamp / 1000)) AS year,
  MONTH(FROM_UNIXTIME(timestamp / 1000)) AS month,
  DAY(FROM_UNIXTIME(timestamp / 1000)) AS day,
  HOUR(FROM_UNIXTIME(timestamp / 1000)) AS hour
from retail_rocket.db.events;

SELECT * FROM events_silver;


In [0]:
CREATE OR REPLACE TABLE funnel_metrics AS
SELECT 
  COUNT(CASE WHEN event = 'view' THEN 1 END) AS views,
  COUNT(CASE WHEN event = 'addtocart' THEN 1 END) AS add_to_cart,
  COUNT(CASE WHEN event = 'transaction' THEN 1 END) AS purchases,

  COUNT(CASE WHEN event = 'addtocart' THEN 1 END) * 1.0 /
  COUNT(CASE WHEN event = 'view' THEN 1 END) AS cart_rate,

  COUNT(CASE WHEN event = 'transaction' THEN 1 END) * 1.0 /
  COUNT(CASE WHEN event = 'addtocart' THEN 1 END) AS purchase_rate

FROM events_silver;

In [0]:
SELECT *
FROM funnel_metrics;

In [0]:
CREATE OR REPLACE TABLE events_by_hour AS
SELECT 
  hour,
  event,
  COUNT(*) AS total_events
FROM events_silver
GROUP BY hour, event
ORDER BY hour;

In [0]:
SELECT *
FROM events_by_hour;

In [0]:
CREATE OR REPLACE TABLE top_products AS
SELECT 
  itemid,
  COUNT(CASE WHEN event = 'view' THEN 1 END) AS views,
  COUNT(CASE WHEN event = 'addtocart' THEN 1 END) AS adds,
  COUNT(CASE WHEN event = 'transaction' THEN 1 END) AS purchases
FROM events_silver
GROUP BY itemid
ORDER BY purchases DESC;

In [0]:
SELECT *
FROM top_products;

In [0]:
USE retail_rocket.db;
CREATE OR REPLACE TABLE ts_purchases AS
SELECT 
  DATE(event_time) AS date,
  COUNT(*) AS purchases
FROM retail_rocket.db.events_silver
WHERE event = 'transaction'
GROUP BY DATE(event_time)
ORDER BY date;
SELECT * FROM ts_purchases;